<a href="https://colab.research.google.com/github/Maxxx-VS/IMA_SibADI/blob/main/ML_2_4_%D0%9E%D0%B1%D1%83%D1%87%D0%B5%D0%BD%D0%B8%D0%B5_%D0%BC%D0%BD%D0%BE%D0%B3%D0%BE%D1%81%D0%BB%D0%BE%D0%B9%D0%BD%D0%BE%D0%B9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Задание 2.4. Обучение многослойной нейросети с 2 входами

In [1]:
!pip install -q onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 15.3 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv('/content/multiregress-092022.csv')
df = df.drop('i', axis=1)
x = df[['Me', 'ne']].to_numpy()
y = df.tc.to_numpy()
n = y.size

In [3]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

scaler = StandardScaler(with_mean=True, with_std=True)
x_s = scaler.fit_transform(x)
y = y.reshape((-1, 1))

In [4]:
import torch
from torch import nn


class MultiLayer(nn.Module):
    def __init__(self):
        super(MultiLayer, self).__init__()

        self.a_hidd = nn.Linear(bias=True, in_features=2, out_features=5)
        self.f_hidd = nn.Tanh()
        self.a_out = nn.Linear(bias=True, in_features=5, out_features=1)

    def forward(self, x):
        a_hidd = self.a_hidd(x)
        z_hidd = self.f_hidd(a_hidd)
        out = self.a_out(z_hidd)
        return out

In [5]:
model = MultiLayer()

In [6]:
x_torch = torch.as_tensor(x_s).float()
y_torch = torch.as_tensor(y).float()

In [7]:
from torch import optim
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [8]:
n_epochs = 500
losses = []

for epoch in range(n_epochs):
    loss = 0.0
    model.train()
    optimizer.zero_grad()
    y_hat_torch = model(x_torch)
    loss = loss_fn(y_hat_torch, y_torch)
    loss.backward()
    optimizer.step()
    losses.append(loss.detach().numpy())
    if epoch % 50 == 0:
        print('Epoch: {}, Loss: {:.2f}'.format(epoch, loss))

Epoch: 0, Loss: 6728.11
Epoch: 50, Loss: 2.34
Epoch: 100, Loss: 2.16
Epoch: 150, Loss: 1.93
Epoch: 200, Loss: 1.38
Epoch: 250, Loss: 0.92
Epoch: 300, Loss: 0.87
Epoch: 350, Loss: 0.82
Epoch: 400, Loss: 0.77
Epoch: 450, Loss: 0.72


In [9]:
print('Скрытый слой :\n Смещения ', model.a_hidd.bias, '\n Веса ', model.a_hidd.weight)
print('Выходной слой :\n Смещения ', model.a_out.bias, '\n Веса ', model.a_out.weight)

Скрытый слой :
 Смещения  Parameter containing:
tensor([ 1.9533,  2.8320, -2.6069,  3.1258, -3.0670], requires_grad=True) 
 Веса  Parameter containing:
tensor([[ 0.5452,  0.2181],
        [ 0.5599,  0.0602],
        [-0.6023, -0.0359],
        [ 0.4640,  0.0422],
        [-0.4961, -0.0680]], requires_grad=True)
Выходной слой :
 Смещения  Parameter containing:
tensor([18.1581], requires_grad=True) 
 Веса  Parameter containing:
tensor([[ 11.2762,  12.9537, -12.0126,  14.3835, -14.3780]],
       requires_grad=True)


In [10]:
import plotly.graph_objects as go
fig3 = go.Figure()
fig3.add_trace(go.Scatter(y=losses,
                          mode='markers+lines', name='loss',
                          marker=dict(color='red', size=3, opacity=0.8)))
fig3.update_layout(title_text="MSE vs epoch", title_font_size=20,
                   xaxis_title="epoch", yaxis_title="MSE")
fig3.show()

In [11]:
import warnings

model.eval()
with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    _ = torch.onnx.export(
        model, x_torch[:1], "multylayer.onnx",
        input_names=['x'], output_names=['y'])
print("Модель сохранена: multylayer.onnx")

[torch.onnx] Obtain model graph for `MultiLayer([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MultiLayer([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Модель сохранена: multylayer.onnx
